# Задание 1

In [1]:
import torch, torch.nn.functional as F

In [2]:
def sliding_window_attention(Q, K, V, window_size):
    # Q,K,V формы (seq_len, batch, d_model)
    seq_len = Q.size(0) 

    # Создаём маску формата (seq_len, seq_len) с -inf для запрещённых связей
    mask = torch.full((seq_len, seq_len), float('-inf'))
    for i in range(seq_len):
        left = max(0, i - window_size)
        right = min(seq_len, i + window_size + 1)
        mask[i, left:right] = 0

    # Переставляем размерности, чтобы батч был первым
    # Ваш код здесь
    Q, K, V = Q.transpose(0, 1), K.transpose(0, 1), V.transpose(0, 1) # shape: (batch, seq_len, d_model)
    
    # Изменяем размер маски для соответствия батч размеру Q (shape: [batch, seq_len, seq_len])
    print(f'mask.shape = {mask.shape}')
    mask = mask.unsqueeze(0).expand(Q.size(0), -1, -1) # Ваш код здесь
    print(f'mask.shape = {mask.shape}')

    # Используем scaled_dot_product_attention с маской
    attn_out = F.scaled_dot_product_attention(Q, K, V, attn_mask=mask) # Ваш код здесь

    # Возвращаем результат обратно в исходном порядке: (seq_len, batch, d_model)
    attn_out = attn_out.transpose(0, 1) # Ваш код здесь

    return attn_out, mask

In [3]:
# Проверка маскированного внимания:
seq_len, batch, d_model = 10, 2, 16 # Ваш код здесь
Q = torch.randn(seq_len, batch, d_model) # Ваш код здесь
out, mask = sliding_window_attention(Q, Q, Q, window_size=2)
print(out.shape)

mask.shape = torch.Size([10, 10])
mask.shape = torch.Size([2, 10, 10])
torch.Size([10, 2, 16])


In [4]:
out.shape, mask.shape

(torch.Size([10, 2, 16]), torch.Size([2, 10, 10]))

In [5]:
# Дополнительная проверка визуализацией
print("\nПример маски для позиции 5 (первый батч):")
print(mask[0, 5].detach().numpy().round(1)) 


Пример маски для позиции 5 (первый батч):
[-inf -inf -inf   0.   0.   0.   0.   0. -inf -inf]


# Theory testing

In [6]:
import torch, torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel

# Параметры теста
batch, heads, seq_len, head_dim = 8, 4, 512, 32
dtype = torch.float16
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Случайные Q,K,V
query = torch.randn(batch, heads, seq_len, head_dim, device=device, dtype=dtype)
key   = torch.randn_like(query)
value = torch.randn_like(query)

# Функция для бенчмарка (в мкс)
import torch.utils.benchmark as benchmark
def bench(f, *args):
    t0 = benchmark.Timer(stmt="f(*args)", globals={"f": f, "args": args})
    return t0.blocked_autorange().mean * 1e6

print("Тестовый запуск...")

# Обычная реализация (Math)
with sdpa_kernel(SDPBackend.MATH):
    t_math = bench(F.scaled_dot_product_attention, query, key, value)
print(f"Math implementation: {t_math:.1f} мкс")

# Flash Attention реализация
with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    try:
        t_flash = bench(F.scaled_dot_product_attention, query, key, value)
        print(f"Flash Attention impl.: {t_flash:.1f} мкс")
    except RuntimeError as e:
        print("Flash Attention не поддерживается:", e)

# Память-эффективная реализация
with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
    t_eff = bench(F.scaled_dot_product_attention, query, key, value)
print(f"Memory-Efficient impl.: {t_eff:.1f} мкс")

Тестовый запуск...
Math implementation: 2177.5 мкс
Flash Attention impl.: 74.7 мкс
Memory-Efficient impl.: 133.6 мкс


# Task 2

In [7]:
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer,BertConfig, BertForSequenceClassification
from torch.optim import AdamW
import tqdm

/home/russele7/practicum/dle/sprint_5/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
imdb = load_dataset("imdb")

In [23]:
# Выберем 1000 примеров для обучения и 500 для теста
train_dataset = imdb['train'].shuffle(seed=42)#.select(range(1000))
test_dataset  = imdb['test'].shuffle(seed=42)#.select(range(500))

In [26]:
# Вывод первых 3 рецензий и меток
for i in range(10000, 10005):
    print(f"Рецензия {i+1}: {train_dataset[i]['text']}")
    print(f"Метка {i+1}: {train_dataset[i]['label']}\n") 

Рецензия 10001: I am sorry to say that this film is indeed bad. It reminds me of a c-grade porn movie with one major difference: no porn.<br /><br />The story and dialogue needs a complete overhaul. Maybe then the bad acting would not have been as noticeable. At the very least, the pacing should have been picked up.<br /><br />While I accept that this had a low budget and the director did a good job visually given what little resources he had, he should have spent more time on the story or better yet, get someone else to write it. Many of the action scenes were just pointless.<br /><br />It was a complete waste of my time.
Метка 10001: 0

Рецензия 10002: Like other people who commented on "Fräulein Doktor" I stumbled by chance upon this little gem on late-night TV without having heard of it before. The strange mixture of a pulp fiction story about a sexy but unscrupulous anti-heroine on the one hand and a realistic and well-researched portrayal of war in the trenches on the other hand 

# Task 3

In [27]:
tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased') # Ваш код здесь

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 57555e16-497e-4ff4-85d5-8e3b8b6f4d79)')' thrown while requesting HEAD https://huggingface.co/google-bert/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


In [28]:

def tokenize_batch(batch):
    texts  = [x['text'] for x in batch]
    labels = [x['label'] for x in batch]
    encoding = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors='pt'
    )
    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': torch.tensor(labels)
    }


In [29]:
def create_dataloader(dataset, batch_size=8, shuffle=True):
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=tokenize_batch
    )

In [30]:
train_loader = create_dataloader(train_dataset)
test_loader  = create_dataloader(test_dataset, shuffle=False)

# Task 4

In [31]:
# Устройство: GPU, если доступна
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Конфигурация
config = BertConfig.from_pretrained('google-bert/bert-base-uncased', num_labels=2)

In [32]:
device

device(type='cuda')

In [33]:
model  = BertForSequenceClassification.from_pretrained(
    'google-bert/bert-base-uncased', config=config
).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
# Оптимизатор
optimizer = AdamW(model.parameters(), lr=2e-5)  # Ваш код здесь 

In [35]:
#

# Task 5

In [36]:
from tqdm.auto import tqdm

In [37]:
epochs = 2
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        # Ваш код здесь
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss   += loss.item()
        preds         = outputs.logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = total_correct / total_samples 

    model.eval()
    val_loss, val_correct, val_samples = 0, 0, 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch} [Eval]"):
            # Ваш код здесь
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(
                input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            val_loss     += outputs.loss
            preds         = outputs.logits.argmax(dim=1)
            val_correct  += (preds == labels).sum().item()
            val_samples  += labels.size(0)


    val_loss = val_loss / len(test_loader)
    val_acc  = val_correct / val_samples

    print(f"\nEpoch {epoch} results:")
    print(f"  Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
    print(f"  Eval : loss={val_loss:.4f}, acc={val_acc:.4f}\n")

print(f"Final Eval Accuracy: {val_acc:.4f}") 

Epoch 1 [Eval]: 100%|██████████| 3125/3125 [06:02<00:00,  8.61it/s]



Epoch 1 results:
  Train: loss=0.2661, acc=0.8907
  Eval : loss=0.2023, acc=0.9174



Epoch 2 [Eval]: 100%|██████████| 3125/3125 [05:55<00:00,  8.78it/s]


Epoch 2 results:
  Train: loss=0.1386, acc=0.9502
  Eval : loss=0.2290, acc=0.9158

Final Eval Accuracy: 0.9158
